# Tier 4.1 — Agentic CRAG with Hybrid First Stage (Azure GPU)

**BSARD RAG Thesis | RQ1 | Evaluator: LLaMA 3.1 8B collective judgment**

## This run: CRAG on hybrid_rrf_k60 first stage (test split)

One CRAG experiment on the **test split** (222 questions), using the T4.0-hybrid first stage
(`hybrid_rrf_k60`: BM25 k1=1.5 b=0.25 lemmatize text_only + mE5-large concat_2x, RRF k=60,
first_stage_k=100). Same collective LLM evaluator (LLaMA 3.1 8B, eval_k=20) and same CRAG loop
parameters as `crag_bm25_test_v2`. First stage is the only axis of variation.

| Experiment | Backbone | Purpose |
|---|---|---|
| `crag_hybrid_rrf_k60_test_v2` | `hybrid_rrf_k60` | CRAG on T4.0-hybrid first stage — ablation |

## Ablation design

| Comparison | Measures |
|---|---|
| T4.1-hybrid vs `hybrid_rrf_k60` (T3-A) | Value of CRAG correction loop on hybrid pool |
| T4.1-hybrid vs `crag_bm25_test_v2` (T4.1-BM25) | Pool quality: same CRAG loop, hybrid vs BM25 first stage |
| T4.1-hybrid vs T4.0-hybrid | Agentic value: CRAG vs non-agentic LLM re-ranking on same hybrid pool |

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC4as_T4_v3`
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. **Prerequisites in blob storage** (upload locally before running):
   - `embeddings/intfloat_multilingual_e5_large_concat_2x.npy` (~87 MB)
   - `embeddings/intfloat_multilingual_e5_large_concat_2x_ids.npy` (~0.2 MB)
   - `results/hybrid/hybrid_rrf_k60_test.json` (significance anchor — T3-A)
   - `results/agentic/CRAG/crag_bm25_test_v2.json` (significance anchor — T4.1-BM25)
   - `results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_hybrid_rrf_k60_test.json` (T4.0-hybrid anchor)
4. Run cells top to bottom

## Resuming after interruption

The experiment script writes a per-question checkpoint (`.ckpt.json`) after every 5 fresh
questions and always on exit — even on `KeyboardInterrupt`. On resume, completed questions
are served from the checkpoint instantly (no LLM calls repeated). Re-run from Cell 0.

## Expected execution times (T4 GPU)

| Phase | Expected time |
|---|---|
| Setup (Cells 0–7) | ~25 min (embedding load adds ~1 min vs BM25-only) |
| CRAG hybrid test run (Cell 9) | ~64–70 min |
| Significance tests (Cell 12) | ~5 min |
| Upload to blob (Cell 14) | ~1 min |
| **Total** | **~95 min** |

## Historical record — three-stage query rewrite (as it actually ran)

**Added retroactively for historical accuracy.** This cell documents the CRAG mechanism that
actually produced `crag_hybrid_rrf_k60_test_v2` — the **canonical T4.1 result** reported in the
thesis (R@10 = 0.4263, R@100 = 0.6542, significant vs. the T3-A hybrid baseline, *p* = 0.046).

Cells below clone the repo at run time and invoke `CRAGRetriever` from
`retrieval/agentic/crag.py` via `scripts/evaluation/tier4/run_crag_experiments.py`. **The
`crag.py` / `prompts.py` currently on `main` have since been simplified** (single per-document
binary evaluator, one rewrite prompt fired at every iteration) and no longer reflect the
collective evaluator + three-stage rewrite loop that generated this notebook's results. This
cell is a self-contained record of that original mechanism, independent of the current shared
module. Source: `TIER41_AGENTIC_CRAG_PLAN.md` §2.2/§2.4/§2.5/§4.1 (design doc, contemporaneous
with the run) plus the two evaluator prompts, which are still present verbatim in
`retrieval/agentic/llm_eval_prompts.py` today.

### Loop mechanism

- **`evaluate`** — one **collective** sufficiency call per iteration (all `eval_k=20` articles
  judged together, not one call per article):
  - **Iteration 0** → `LLM_JUDGE_COLLECTIVE_STRICT_PROMPT` ("Variant B") — strict.
  - **Iterations 1+** → `LLM_JUDGE_COLLECTIVE_PROMPT` ("Variant D") — progressively relaxed;
    a single relevant article suffices for single-article questions.
  - `Oui` → Correct → `finalize`. `Non` → Incorrect → `rewrite_query`.
- **`rewrite_query`** — three-stage, moving from a literal statutory rephrasing to
  aspect-focused reformulations:
  1. **Iteration 0 → 1 (Incorrect at iter 0):** standard statutory rewrite —
     `CRAG_REWRITE_PROMPT(original_query, current_query)` → plain rewritten query.
  2. **Iteration 1 → 2 (Incorrect at iter 1):** fires **twice** —
     `CRAG_ASPECT_EXTRACT_PROMPT(original_query)` first, extracting 3–5 legal-aspect labels
     from the original question into `question_aspects`; then
     `CRAG_REWRITE_FOCUSED_PROMPT(original_query, previous_aspects_block, target_aspect=aspects[0])`
     → a query focused on the first aspect. `aspect_index` set to 1.
  3. **Iterations 2+ (still Incorrect):** `CRAG_REWRITE_FOCUSED_PROMPT` again, targeting
     `question_aspects[aspect_index % len(aspects)]`, `aspect_index` incremented each time —
     cycling through the extracted aspects on later iterations.
- **Loop termination:** `Correct`, or `iteration >= max_iterations` (=6, never hit — 0.0%
  limit-terminated on both backbones), or the rewrite returns an unchanged query.

### Verbatim prompt text (recovered — unchanged in the current repo)

**`LLM_JUDGE_COLLECTIVE_STRICT_PROMPT`** (iteration 0 evaluator — `retrieval/agentic/llm_eval_prompts.py`):
```
Question : {question}

Passages :
{articles_block}

Au moins un des passages ci-dessus est-il pertinent pour répondre à la question ?
Répondez uniquement par « Oui » ou « Non ».

Pertinent :
```

**`LLM_JUDGE_COLLECTIVE_PROMPT`** (iterations 1+ evaluator, relaxed — `retrieval/agentic/llm_eval_prompts.py`):
```
Question : {question}

Passages :
{articles_block}

Parmi les passages ci-dessus, y en a-t-il au moins un qui contient des informations utiles pour répondre à la question ?
Répondez uniquement par « Oui » ou « Non ».

Pertinent :
```

**`CRAG_REWRITE_PROMPT`** (stage 1 — literal statutory rephrasing, fired on Incorrect at
iteration 0 — `retrieval/agentic/prompts.py`):
```
Vous êtes un expert en amélioration de requêtes pour un système de recherche d'articles de droit belge.

Requête originale : {original_query}
Requête actuelle  : {current_query}
Problème : Les articles récupérés sont hors sujet ou non pertinents.

Réécrivez la requête en utilisant une terminologie juridique belge plus précise ou en abordant la question sous un angle différent susceptible de mieux correspondre au vocabulaire des textes de loi belges.

Répondez uniquement avec la requête réécrite, sans explication.

Requête réécrite :
```

### Stages 2 and 3 — exact wording not recoverable

`CRAG_ASPECT_EXTRACT_PROMPT` and `CRAG_REWRITE_FOCUSED_PROMPT` (the aspect-focused
reformulation stages) are **not present anywhere in the current repository or its git
history** — `retrieval/agentic/prompts.py` was already simplified to only
`CRAG_REWRITE_PROMPT` and the unused `CRAG_DECOMPOSE_PROMPT` by the point this component's
history begins, and no earlier revision exists to recover the originals from. The only
surviving record is the functional contract in `TIER41_AGENTIC_CRAG_PLAN.md` §2.4, reproduced
here rather than fabricated as if it were the literal prompt text:

- **`CRAG_ASPECT_EXTRACT_PROMPT`** — placeholder `{question}`. Instructs the model to identify
  3–5 distinct legal dimensions/sub-questions raised by the question, based solely on the
  question itself (not general legal knowledge). Output: a numbered list of compact aspect
  labels (3–6 words each), e.g. `droit de recours administratif`,
  `obligation de motivation de l'acte`, `délai de prescription`.
- **`CRAG_REWRITE_FOCUSED_PROMPT`** — placeholders `{original_query}`,
  `{previous_aspects_block}`, `{target_aspect}`. Instructs the model to write a query that
  specifically targets `{target_aspect}` while avoiding aspects already explored (listed in
  `{previous_aspects_block}`). Output (two-line structured format, parsed by
  `parse_aspect_rewrite()`):
  ```
  Aspect ciblé : <3-6 word label>
  Requête : <plain French query>
  ```

Loop-behaviour telemetry from the result JSON is consistent with this three-stage design:
on the canonical hybrid run, only 3.6% of queries were ever rewritten and mean iterations/query
was 1.04 (§4.4/§4.5 of the plan doc) — the strong hybrid first stage rarely reaches past stage 1.

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List + Write → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)

In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',
            'OLLAMA_FLASH_ATTENTION': '1',
            'OLLAMA_HOST': '0.0.0.0:11434',
            'OLLAMA_NUM_CTX': '20000',
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)

    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')

In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
# Resumes partial downloads automatically (seeks to existing byte offset).
# Missing optional blobs show a warning, not an error.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob', 'tqdm'],
               check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR  = Path(REPO_DIR) / 'output'
EMB_DIR     = OUTPUT_DIR / 'embeddings'
RESULTS_DIR = OUTPUT_DIR / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'hybrid').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'agentic' / 'CRAG').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank').mkdir(parents=True, exist_ok=True)

EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)


def _download_blob(blob_name: str, dest_path: Path, required: bool = True) -> bool:
    """Download blob → dest_path with tqdm progress and byte-level resume."""
    try:
        bc         = client.get_blob_client(blob_name)
        total_size = bc.get_blob_properties()['size']
    except Exception:
        tag = '[REQUIRED]' if required else '[optional]'
        print(f'  {tag} {blob_name} — not found in blob storage')
        if required:
            raise FileNotFoundError(f'Required blob missing: {blob_name}')
        return False

    existing = dest_path.stat().st_size if dest_path.exists() else 0
    if existing == total_size:
        print(f'  Already complete: {dest_path.name} ({total_size / 1e6:.1f} MB)')
        return True

    offset = existing if 0 < existing < total_size else 0
    mode   = 'ab' if offset > 0 else 'wb'
    if offset > 0:
        print(f'  Resuming {dest_path.name}: {offset/1e6:.1f}/{total_size/1e6:.1f} MB done')

    with tqdm(total=total_size, initial=offset, unit='B', unit_scale=True,
              desc=f'  {dest_path.name}', ncols=90, leave=True) as pbar:
        with open(dest_path, mode) as f:
            for chunk in bc.download_blob(offset=offset).chunks():
                f.write(chunk)
                pbar.update(len(chunk))
    return True


# ── Required: corpus + mE5-large (non-instruct) embeddings ───────────────────
print('=== Required files ===')
_download_blob('bsard_articles_dedup.parquet', OUTPUT_DIR / 'bsard_articles_dedup.parquet')
_download_blob('bsard_corpus.db',              OUTPUT_DIR / 'bsard_corpus.db')
_download_blob(f'embeddings/{EMB_SLUG}.npy',      EMB_DIR / f'{EMB_SLUG}.npy')
_download_blob(f'embeddings/{EMB_SLUG}_ids.npy',  EMB_DIR / f'{EMB_SLUG}_ids.npy')

# ── Optional: significance anchors + prior checkpoint ────────────────────────
print('\n=== Optional files (warnings only if missing) ===')
_download_blob(
    'results/hybrid/hybrid_rrf_k60_test.json',
    RESULTS_DIR / 'hybrid' / 'hybrid_rrf_k60_test.json',
    required=False,
)
_download_blob(
    'results/agentic/CRAG/crag_bm25_test_v2.json',
    RESULTS_DIR / 'agentic' / 'CRAG' / 'crag_bm25_test_v2.json',
    required=False,
)
_download_blob(
    'results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_hybrid_rrf_k60_test.json',
    RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank' /
    'llm_rerank_binary_top50_hybrid_rrf_k60_test.json',
    required=False,
)
# Download prior checkpoint if interrupted on a previous VM session
_download_blob(
    'crag_hybrid_rrf_k60_test_v2.ckpt.json',
    OUTPUT_DIR / 'results' / 'agentic' / 'CRAG' / 'crag_hybrid_rrf_k60_test_v2.ckpt.json',
    required=False,
)

print('\nAll downloads complete.')

In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys, os

cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
cuda_ver = 'cu118' if 'release 11' in cuda_out else 'cu121'
print(f'CUDA detected → torch variant: {cuda_ver}')

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
      '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}'],
     'torch'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'],
     'tf-keras'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.2'],
     'timm>=0.9.2'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'],
     'spacy'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package from the RQ3 repo
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

In [ ]:
# ── Cell 6: Install spaCy French model ───────────────────────────────────────
# BM25 lemmatization (used in the hybrid first stage) requires fr_core_news_lg.
import subprocess, sys
import spacy as _spacy

_sv   = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    _sv2  = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'spaCy fr_core_news_lg install failed:\n{r.stderr[-400:]}')

import spacy
nlp = spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# ── Cell 7: Pre-flight checks + LLM latency benchmark ────────────────────────
import json, os, sys, time
import requests as _req
from pathlib import Path

os.chdir(REPO_DIR)

# ── File checks ───────────────────────────────────────────────────────────────
EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
for p in [
    Path('output/bsard_articles_dedup.parquet'),
    Path('output/bsard_corpus.db'),
    Path('evaluation/data/fewshot_examples.json'),
    Path(f'output/embeddings/{EMB_SLUG}.npy'),
    Path(f'output/embeddings/{EMB_SLUG}_ids.npy'),
]:
    print(f'  {"OK     " if p.exists() else "MISSING"}  {p}')

# ── Ollama alive ──────────────────────────────────────────────────────────────
try:
    r      = _req.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'\nOllama alive. Models: {models}')
except Exception as e:
    raise RuntimeError(f'Ollama not reachable: {e}')

# ── LLM latency benchmark — collective prompt (eval_k=20 simulation) ──────────
SAMPLE_ARTICLES = '\n\n'.join(
    f'--- Article {i+1} ---\n'
    'Les travailleurs salariés ont droit à un congé parental pour s\'occuper d\'un enfant. '
    'Les conditions d\'octroi sont fixées par la convention collective de travail. '
    'La durée maximale est déterminée en fonction de l\'ancienneté du travailleur dans l\'entreprise.'
    for i in range(20)
)
COLLECTIVE_PROMPT = (
    'Question : Quelles sont les conditions pour obtenir un congé parental en Belgique ?\n\n'
    f'{SAMPLE_ARTICLES}\n\n'
    'Ces articles fournissent-ils collectivement une base juridique suffisante pour répondre '
    'à cette question de manière complète et précise ?\n'
    'Répondez par « Oui » ou « Non » en premier, suivi d\'une brève justification.\n\nRéponse :'
)

def llm_generate(prompt, max_tokens=80):
    t0   = time.perf_counter()
    resp = _req.post('http://localhost:11434/api/generate', json={
        'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
        'options': {'temperature': 0.0, 'num_predict': max_tokens},
    }, timeout=300)
    return resp.json()['response'].strip(), (time.perf_counter() - t0) * 1000

print('\nBenchmarking collective LLM call (eval_k=20 simulation, 3 calls)...')
lats = []
for i in range(3):
    resp, lat = llm_generate(COLLECTIVE_PROMPT, max_tokens=80)
    lats.append(lat)
    print(f'  Call {i+1}: {resp[:50].replace(chr(10), " ")!r:52s}  {lat:.0f} ms')

warm_lat = sum(lats[1:]) / 2
mean_lat = sum(lats) / len(lats)
print(f'\nWarm mean (calls 2–3): {warm_lat:.0f} ms  |  Overall mean: {mean_lat:.0f} ms')

if warm_lat < 10_000:
    print('GPU confirmed (fast).')
elif warm_lat < 60_000:
    print('MARGINAL — verify GPU in Cell 1.')
else:
    print('WARNING: likely on CPU. Re-check nvidia-smi.')

# ── Runtime estimate (same scenario model as BM25 run) ────────────────────────
_RETRIEVAL_MS   = 2_000
_REWRITE_STD_MS = 2_800
_REWRITE_ASP_MS = 10_000
n_test = 222

p_a = 0.50; p_b = 0.42; p_c = 0.08
lat_a = _RETRIEVAL_MS + warm_lat
lat_b = 2*_RETRIEVAL_MS + warm_lat + _REWRITE_STD_MS + warm_lat
lat_c = 3*_RETRIEVAL_MS + warm_lat + _REWRITE_STD_MS + warm_lat + _REWRITE_ASP_MS + warm_lat
mean_query_ms = p_a*lat_a + p_b*lat_b + p_c*lat_c
est_min = (n_test * mean_query_ms / 1_000) / 60.0
print(f'\nEstimated total runtime (222 questions, fresh): ~{est_min:.0f} min')
print('(hybrid first-stage latency ~57 ms/q is negligible vs LLM eval time)')

---
## Primary v2 Hybrid Run — hybrid_rrf_k60 First Stage, Test Split

Collective LLM evaluator (`eval_k=20`, single sufficiency call per iteration).
Progressive criterion relaxation: iteration 0 = strict Variant B, iterations 1+ = lenient Variant D.
Hybrid first stage: BM25 (k1=1.5, b=0.25, lemmatize, text_only) + mE5-large concat_2x, RRF k=60.
All CRAG loop parameters identical to `crag_bm25_test_v2`.

**Resumability:** the script writes a `.ckpt.json` after every 5 fresh questions and always on
clean exit. Re-running Cell 9 after interruption resumes from the last checkpoint — no LLM
calls are repeated. Upload the checkpoint to blob (Cell 14) before stopping the VM so it
survives to the next session.

In [ ]:
# ── Cell 8: eval_k configuration ─────────────────────────────────────────────
# eval_k=20 locked (aligned-run value — 95.9% Correct first-pass for BM25).
# Unchanged from crag_bm25_test_v2 — first stage is the only axis of variation.

EVAL_K         = 20    # docs judged per collective LLM call
MAX_ITERATIONS = 6     # hard loop limit (0.0% limit-terminated in BM25 run)
MAX_ART_TOKENS = 400   # per-article token budget → ~8,000 total for eval_k=20
BACKBONE_TOP_K = 100   # retrieve this many before evaluate + finalize

EXPERIMENT_ID = f'crag_hybrid_rrf_k60_test_v2'
RESULTS_DIR_PATH = f'{REPO_DIR}/output/results/agentic/CRAG'

print(f'experiment_id  = {EXPERIMENT_ID}')
print(f'eval_k         = {EVAL_K}')
print(f'max_iterations = {MAX_ITERATIONS}')
print(f'max_art_tokens = {MAX_ART_TOKENS}')
print(f'backbone_top_k = {BACKBONE_TOP_K}')

# ── Show checkpoint state ─────────────────────────────────────────────────────
import json
from pathlib import Path

result_path = Path(f'{RESULTS_DIR_PATH}/{EXPERIMENT_ID}.json')
ckpt_path   = Path(f'{RESULTS_DIR_PATH}/{EXPERIMENT_ID}.ckpt.json')

if result_path.exists():
    r10 = json.loads(result_path.read_text())['metrics'].get('Recall@10', 0)
    print(f'\n✓ RESULT EXISTS — Cell 9 will skip.  R@10={r10:.4f}')
elif ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text(encoding='utf-8'))
    n_done = len(ckpt)
    print(f'\n→ CHECKPOINT FOUND: {n_done}/222 questions done — Cell 9 will resume from here.')
    import time as _t
    avg_lat_s = 17.3  # empirical from BM25 run
    eta_min = avg_lat_s * (222 - n_done) / 60
    print(f'  Estimated remaining time: ~{eta_min:.0f} min')
else:
    print('\n→ NO CHECKPOINT — Cell 9 will start fresh (222 questions).')

In [ ]:
# ── Cell 9: CRAG hybrid test run — crag_hybrid_rrf_k60_test_v2 ────────────────
# Backbone: hybrid_rrf_k60 (BM25 k1=1.5 b=0.25 lem text_only + mE5-large concat_2x, RRF k=60)
# Writes: output/results/agentic/CRAG/crag_hybrid_rrf_k60_test_v2.json
# Expected: ~64–70 min (eval_k=20, max_iterations=6)
#
# RESUMABILITY
# ─────────────
# The script writes a per-question checkpoint every 5 fresh questions AND on clean exit.
# Progress is printed after each question:
#   [  7/222]  R@10=0.3456  lat=18.2s  avg=17.4s/q  elapsed=2.0m  ETA≈60m
# On resume, checkpointed questions are skipped instantly — no LLM calls repeated.
#
# INTERRUPT RECOVERY
# ───────────────────
# 1. Interrupt the cell (■ stop button)
# 2. Upload checkpoint to blob (Cell 14)
# 3. Restart VM if needed; re-run Cells 0–8
# 4. Re-run this cell — it resumes from the checkpoint automatically
import json, os, subprocess, sys, time
from pathlib import Path

CRAG_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
CRAG_DIR.mkdir(parents=True, exist_ok=True)
OUT = CRAG_DIR / 'crag_hybrid_rrf_k60_test_v2.json'

if OUT.exists():
    r10 = json.loads(OUT.read_text())['metrics'].get('Recall@10', 0)
    print(f'Already exists — skipping.  R@10={r10:.4f}')
else:
    pull = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True
    )
    print('git pull:', pull.stdout.strip() or pull.stderr.strip())

    print(f'Running CRAG hybrid_rrf_k60 test (~64-70 min, eval_k={EVAL_K})...')
    print('Progress is printed after each fresh question.')
    print('Checkpoint is saved every 5 questions and on exit.')
    print()
    t0 = time.time()
    # stdout/stderr not captured — progress streams live to this cell's output
    result = subprocess.run(
        [sys.executable, 'scripts/evaluation/tier4/run_crag_experiments.py',
         '--split', 'test', '--variant', 'hybrid_rrf_k60',
         '--max-iterations', str(MAX_ITERATIONS),
         '--eval-k', str(EVAL_K),
         '--max-article-tokens', str(MAX_ART_TOKENS)],
        cwd=REPO_DIR, timeout=18000,
    )
    print(f'\nDone in {(time.time()-t0)/60:.1f} min (exit {result.returncode})')
    if result.returncode != 0:
        raise RuntimeError('Hybrid CRAG experiment failed — check output above')

In [ ]:
# ── Cell 10: Review hybrid result ─────────────────────────────────────────────
import json
from pathlib import Path

CRAG_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
path = CRAG_DIR / 'crag_hybrid_rrf_k60_test_v2.json'

if not path.exists():
    print('crag_hybrid_rrf_k60_test_v2.json not found — run Cell 9 first.')
else:
    r    = json.loads(path.read_text())
    m    = r.get('metrics', {})
    loop = r.get('crag_loop_stats', {})
    bd   = r.get('latency_breakdown_ms_mean', {})
    lim  = loop.get('fraction_queries_terminated_by_limit', 0)

    print(f'{"Metric":<25} {"Value":>10}')
    print('-' * 38)
    print(f'  {"R@10":<23} {m.get("Recall@10", 0):>10.4f}')
    print(f'  {"R@100":<23} {m.get("Recall@100", 0):>10.4f}')
    print(f'  {"MRR@10":<23} {m.get("MRR@10", 0):>10.4f}')
    print(f'  {"Mean latency (ms)":<23} {r.get("latency_ms_mean", 0):>10.0f}')
    print()

    # Show fresh-only latency if a partial resume occurred
    fresh_lat = r.get('latency_distribution_fresh_only', {})
    if fresh_lat:
        print(f'  (Latency above is fresh-only: {fresh_lat["n_fresh_questions"]} questions,'
              f' mean={fresh_lat["mean_ms"]:.0f} ms)')
        print()

    print('Loop stats:')
    print(f'  Correct first-pass  : {loop.get("fraction_queries_correct_first_pass", 0):.1%}')
    print(f'  Rewritten           : {loop.get("fraction_queries_rewritten", 0):.1%}')
    print(f'  Used aspect rewrite : {loop.get("fraction_queries_used_aspect_rewrite", 0):.1%}')
    print(f'  Terminated by limit : {lim:.1%}')
    print()
    print('Latency breakdown (ms/query):')
    print(f'  retrieval={bd.get("retrieval", 0):.0f}  '
          f'evaluator_llm={bd.get("evaluator_llm", 0):.0f}  '
          f'rewrite_llm={bd.get("rewrite_llm", 0):.0f}')

    if lim > 0.30:
        print(f'\n*** WARNING: {lim:.0%} queries hit max_iterations limit.')

In [ ]:
# ── Cell 11: Per-iteration recall progression ──────────────────────────────────
# Note: covers only fresh questions (not checkpointed ones — those have empty traces).
import json
from pathlib import Path

CRAG_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
path = CRAG_DIR / 'crag_hybrid_rrf_k60_test_v2.json'

if not path.exists():
    print('crag_hybrid_rrf_k60_test_v2.json not found — run Cell 9 first.')
else:
    r   = json.loads(path.read_text())
    pit = r.get('per_iteration_recall', {})

    if not pit:
        print('per_iteration_recall not in result.')
    else:
        print('── Per-iteration recall @ eval_k ────────────────────────────────')
        for k in sorted(k for k in pit if k.startswith('mean_recall_at_eval_k_iter_')):
            iter_idx = k.split('_')[-1]
            print(f'  Iteration {iter_idx} : {pit[k]:.4f}')
        print()

        # Compare to BM25 run if available
        bm25_path = CRAG_DIR / 'crag_bm25_test_v2.json'
        if bm25_path.exists():
            bm25_pit = json.loads(bm25_path.read_text()).get('per_iteration_recall', {})
            if bm25_pit:
                hyb_r0  = pit.get('mean_recall_at_eval_k_iter_0', 0)
                bm25_r0 = bm25_pit.get('mean_recall_at_eval_k_iter_0', 0)
                print('── Comparison: hybrid vs BM25 iter-0 recall @ eval_k ────────────')
                print(f'  Hybrid iter-0 : {hyb_r0:.4f}')
                print(f'  BM25   iter-0 : {bm25_r0:.4f}  (Δ = {hyb_r0 - bm25_r0:+.4f})')
                print()

        n_rw = pit.get('n_queries_with_rewrites', 0)
        print(f'Queries with ≥1 rewrite : {n_rw}')
        if n_rw > 0:
            print(f'  Improved  after rewrite : '
                  f'{pit.get("fraction_improved_after_rewrite",  0):.1%}')
            print(f'  Unchanged after rewrite : '
                  f'{pit.get("fraction_unchanged_after_rewrite", 0):.1%}')
            print(f'  Regressed after rewrite : '
                  f'{pit.get("fraction_regressed_after_rewrite", 0):.1%}')
            print(f'  Mean recall Δ           : '
                  f'{pit.get("mean_recall_delta_after_rewrite", 0):+.4f}')
        print()
        print('Note: recall is measured over the eval_k documents shown to the LLM,')
        print('not over the full top-100 candidate pool used for final ranking.')
        n_fresh = r.get('latency_distribution_fresh_only', {}).get('n_fresh_questions')
        if n_fresh and n_fresh < 222:
            print(f'Note: per-iteration recall covers {n_fresh}/222 fresh questions only.')

In [ ]:
# ── Cell 12: Significance tests (three-way) ───────────────────────────────────
# Primary:     T4.1-hybrid vs hybrid_rrf_k60 (T3-A)  → CRAG loop value on hybrid pool
# Secondary A: T4.1-hybrid vs crag_bm25_test_v2      → pool quality (same CRAG loop)
# Secondary B: T4.1-hybrid vs T4.0-hybrid            → agentic vs non-agentic on hybrid pool
import json, os, sys
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel
from bsard_evaluation import per_query_recall as _bsard_per_query_recall

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

CRAG_DIR   = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
HYBRID_DIR = Path(f'{REPO_DIR}/output/results/hybrid')
LLM_DIR    = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')

HYBRID_CRAG_PATH = CRAG_DIR   / 'crag_hybrid_rrf_k60_test_v2.json'
T3A_PATH         = HYBRID_DIR / 'hybrid_rrf_k60_test.json'
BM25_CRAG_PATH   = CRAG_DIR   / 'crag_bm25_test_v2.json'
T40_HYB_PATH     = LLM_DIR    / 'llm_rerank_binary_top50_hybrid_rrf_k60_test.json'

if not HYBRID_CRAG_PATH.exists():
    print('T4.1-hybrid result not found — run Cell 9 first.')
else:
    hyb_crag = json.loads(HYBRID_CRAG_PATH.read_text(encoding='utf-8'))
    if '_trec_run' not in hyb_crag or '_trec_qrels' not in hyb_crag:
        raise RuntimeError('_trec_run/_trec_qrels missing — re-run with updated runner.py')

    r_hyb_crag_k10  = _bsard_per_query_recall(hyb_crag['_trec_qrels'], hyb_crag['_trec_run'], 10)
    r_hyb_crag_k100 = _bsard_per_query_recall(hyb_crag['_trec_qrels'], hyb_crag['_trec_run'], 100)
    print(f'T4.1-hybrid  R@10 = {np.mean(r_hyb_crag_k10):.4f}  '
          f'R@100 = {np.mean(r_hyb_crag_k100):.4f}')

    hyb_crag.setdefault('secondary_significance', {})

    # ── PRIMARY: T4.1-hybrid vs hybrid_rrf_k60 (T3-A) ─────────────────────────
    print('\n' + '='*60)
    print('PRIMARY: T4.1-hybrid vs hybrid_rrf_k60 (T3-A)')
    print('  Measures: value of CRAG correction loop on hybrid candidate pool')
    if T3A_PATH.exists():
        t3a = json.loads(T3A_PATH.read_text(encoding='utf-8'))
        r_t3a_k10  = _bsard_per_query_recall(t3a['_trec_qrels'], t3a['_trec_run'], 10)
        r_t3a_k100 = _bsard_per_query_recall(t3a['_trec_qrels'], t3a['_trec_run'], 100)
        if r_t3a_k10 is not None:
            _, p10  = ttest_rel(r_hyb_crag_k10,  r_t3a_k10)
            _, p100 = ttest_rel(r_hyb_crag_k100, r_t3a_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10 < 0.05 else 'not significant'
            print(f'  T4.1-hybrid         R@10 = {np.mean(r_hyb_crag_k10):.4f}')
            print(f'  hybrid_rrf_k60 T3-A R@10 = {np.mean(r_t3a_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_hyb_crag_k10) - np.mean(r_t3a_k10):+.4f}')
            print(f'  Delta R@100 = {np.mean(r_hyb_crag_k100) - np.mean(r_t3a_k100):+.4f}')
            print(f'  p-value R@10  = {p10:.4f}  → {sig_str}')
            print(f'  p-value R@100 = {p100:.4f}')
            hyb_crag['significance_vs_anchor'] = {
                'anchor_experiment_id': 'hybrid_rrf_k60_test',
                'p_value_recall10':  round(float(p10),  4),
                'p_value_recall100': round(float(p100), 4),
                'significant':       bool(p10 < 0.05),
            }
    else:
        print(f'  [WARN] {T3A_PATH.name} not found — download from blob storage')

    # ── SECONDARY A: T4.1-hybrid vs crag_bm25_test_v2 ────────────────────────
    print('\nSECONDARY A: T4.1-hybrid vs crag_bm25_test_v2 (T4.1-BM25)')
    print('  Measures: pool quality — same CRAG loop, hybrid vs BM25 first stage')
    if BM25_CRAG_PATH.exists():
        bm25_crag = json.loads(BM25_CRAG_PATH.read_text(encoding='utf-8'))
        r_bm25_k10  = _bsard_per_query_recall(bm25_crag['_trec_qrels'], bm25_crag['_trec_run'], 10)
        r_bm25_k100 = _bsard_per_query_recall(bm25_crag['_trec_qrels'], bm25_crag['_trec_run'], 100)
        if r_bm25_k10 is not None:
            _, p10s  = ttest_rel(r_hyb_crag_k10,  r_bm25_k10)
            _, p100s = ttest_rel(r_hyb_crag_k100, r_bm25_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10s < 0.05 else 'not significant'
            print(f'  T4.1-hybrid  R@10 = {np.mean(r_hyb_crag_k10):.4f}')
            print(f'  T4.1-BM25    R@10 = {np.mean(r_bm25_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_hyb_crag_k10) - np.mean(r_bm25_k10):+.4f}')
            print(f'  p-value R@10  = {p10s:.4f}  → {sig_str}')
            hyb_crag['secondary_significance']['vs_crag_bm25_test_v2'] = {
                'anchor_experiment_id': 'crag_bm25_test_v2',
                'p_value_recall10':  round(float(p10s),  4),
                'p_value_recall100': round(float(p100s), 4),
                'significant':       bool(p10s < 0.05),
            }
    else:
        print(f'  [WARN] {BM25_CRAG_PATH.name} not found — download from blob storage')

    # ── SECONDARY B: T4.1-hybrid vs T4.0-hybrid ───────────────────────────────
    print('\nSECONDARY B: T4.1-hybrid vs T4.0-hybrid')
    print('  Measures: agentic value — CRAG loop vs non-agentic LLM re-ranking, same hybrid pool')
    if T40_HYB_PATH.exists():
        t40_hyb = json.loads(T40_HYB_PATH.read_text(encoding='utf-8'))
        r_t40_k10  = _bsard_per_query_recall(t40_hyb['_trec_qrels'], t40_hyb['_trec_run'], 10)
        r_t40_k100 = _bsard_per_query_recall(t40_hyb['_trec_qrels'], t40_hyb['_trec_run'], 100)
        if r_t40_k10 is not None:
            _, p10t  = ttest_rel(r_hyb_crag_k10,  r_t40_k10)
            _, p100t = ttest_rel(r_hyb_crag_k100, r_t40_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10t < 0.05 else 'not significant'
            print(f'  T4.1-hybrid  R@10 = {np.mean(r_hyb_crag_k10):.4f}')
            print(f'  T4.0-hybrid  R@10 = {np.mean(r_t40_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_hyb_crag_k10) - np.mean(r_t40_k10):+.4f}')
            print(f'  p-value R@10  = {p10t:.4f}  → {sig_str}')
            hyb_crag['secondary_significance']['vs_llm_rerank_binary_top50_hybrid_rrf_k60_test'] = {
                'anchor_experiment_id': 'llm_rerank_binary_top50_hybrid_rrf_k60_test',
                'p_value_recall10':  round(float(p10t),  4),
                'p_value_recall100': round(float(p100t), 4),
                'significant':       bool(p10t < 0.05),
            }
    else:
        print(f'  [WARN] {T40_HYB_PATH.name} not found — download from blob storage')

    HYBRID_CRAG_PATH.write_text(
        json.dumps(hyb_crag, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'\nPatched {HYBRID_CRAG_PATH.name} with significance results.')

In [ ]:
# ── Cell 13: Final results summary ────────────────────────────────────────────
import json
from pathlib import Path

CRAG_DIR   = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
HYBRID_DIR = Path(f'{REPO_DIR}/output/results/hybrid')
LLM_DIR    = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')

rows = [
    ('hybrid_rrf_k60 (T3-A, no rerank)',
     HYBRID_DIR / 'hybrid_rrf_k60_test.json'),
    ('T4.0-hybrid (LLM rerank, binary)',
     LLM_DIR / 'llm_rerank_binary_top50_hybrid_rrf_k60_test.json'),
    ('T4.1-BM25 (CRAG bm25)',
     CRAG_DIR / 'crag_bm25_test_v2.json'),
    ('T4.1-hybrid (CRAG hybrid_rrf_k60)',
     CRAG_DIR / 'crag_hybrid_rrf_k60_test_v2.json'),
]

print(f'{"System":<45}  R@10    R@100   MRR@10  Lat(ms)')
print('=' * 90)
for label, path in rows:
    if not path.exists():
        print(f'  {label:<43}  (not found)')
        continue
    d = json.loads(path.read_text())
    m = d.get('metrics', {})
    lat = d.get('latency_ms_mean', 0)
    print(f'  {label:<43}  {m.get("Recall@10",0):.4f}  '
          f'{m.get("Recall@100",0):.4f}  {m.get("MRR@10",0):.4f}  {lat:.0f}')
print()

for label, path in rows[2:]:
    if not path.exists(): continue
    d    = json.loads(path.read_text())
    loop = d.get('crag_loop_stats', {})
    sig  = d.get('significance_vs_anchor', {})
    if loop:
        print(f'Loop stats ({label}):')
        print(f'  Correct first-pass  : {loop.get("fraction_queries_correct_first_pass", 0):.1%}')
        print(f'  Rewritten           : {loop.get("fraction_queries_rewritten", 0):.1%}')
        print(f'  Terminated by limit : {loop.get("fraction_queries_terminated_by_limit", 0):.1%}')
        if sig:
            anchor = sig.get('anchor_experiment_id', '?')
            p10    = sig.get('p_value_recall10', None)
            sigstr = 'significant' if sig.get('significant') else 'not significant'
            p_str  = f'p={p10:.4f}' if p10 is not None else 'not computed'
            print(f'  Significance vs {anchor}: {sigstr} ({p_str})')
        print()

In [ ]:
# ── Cell 14: Upload results + checkpoint to Azure Blob Storage ─────────────────
# Run this cell before stopping the VM — even mid-run — so the checkpoint survives.
# output/ is gitignored — results are NOT committed to git.
from pathlib import Path
from azure.storage.blob import ContainerClient

CRAG_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')

client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
uploaded = []

for json_file in sorted(CRAG_DIR.glob('crag_hybrid_rrf_k60_*.json')):
    blob_name = f'results/agentic/CRAG/{json_file.name}'
    print(f'  Uploading {json_file.name} → {blob_name} ...', end='', flush=True)
    with open(json_file, 'rb') as f:
        client.get_blob_client(blob_name).upload_blob(f, overwrite=True)
    print(' done')
    uploaded.append(blob_name)

print(f'\nUploaded {len(uploaded)} file(s) to blob storage.')
if not any('crag_hybrid_rrf_k60_test_v2.json' in u and '.ckpt' not in u for u in uploaded):
    print('NOTE: final result JSON not yet present — run Cell 9 to completion.')
print('Download locally via Azure Storage Explorer → output/results/agentic/CRAG/')